In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
# ============================================================
# 1. LOAD
# ============================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train = train.copy()
test = test.copy()


# ============================================================
# 2. BASIC DATA TYPES
# ============================================================

for df in [train, test]:
    df["observation_timestamp"] = pd.to_datetime(
        df["observation_timestamp"]
    )

    # Ensure chronological ordering within each station
    df.sort_values(
        ["station", "observation_timestamp"],
        inplace=True
    )

    df.reset_index(drop=True, inplace=True)


# ============================================================
# 3. SEPARATE TARGET
# ============================================================

target = "PM2_5_next_hour"

y = train[target].copy()

train_features = train.drop(columns=[target]).copy()
test_features = test.copy()


# ============================================================
# 4. REMOVE NON-PREDICTIVE IDENTIFIER
# ============================================================

train_features.drop(columns=["id"], inplace=True)
test_features.drop(columns=["id"], inplace=True)


# ============================================================
# 5. CALENDAR FEATURES
# ============================================================

for df in [train_features, test_features]:

    df["year"] = df["observation_timestamp"].dt.year
    df["month"] = df["observation_timestamp"].dt.month
    df["day"] = df["observation_timestamp"].dt.day
    df["hour"] = df["observation_timestamp"].dt.hour

    df["dayofweek"] = df["observation_timestamp"].dt.dayofweek
    df["dayofyear"] = df["observation_timestamp"].dt.dayofyear

    df["weekofyear"] = (
        df["observation_timestamp"]
        .dt.isocalendar()
        .week
        .astype(int)
    )


# ============================================================
# 6. CYCLICAL TIME FEATURES
# ============================================================

for df in [train_features, test_features]:

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["month_sin"] = np.sin(
        2 * np.pi * df["month"] / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * df["month"] / 12
    )

    df["dayofweek_sin"] = np.sin(
        2 * np.pi * df["dayofweek"] / 7
    )

    df["dayofweek_cos"] = np.cos(
        2 * np.pi * df["dayofweek"] / 7
    )


# ============================================================
# 7. WIND DIRECTION
# ============================================================

wind_angles = {
    "N": 0,
    "NNE": 22.5,
    "NE": 45,
    "ENE": 67.5,
    "E": 90,
    "ESE": 112.5,
    "SE": 135,
    "SSE": 157.5,
    "S": 180,
    "SSW": 202.5,
    "SW": 225,
    "WSW": 247.5,
    "W": 270,
    "WNW": 292.5,
    "NW": 315,
    "NNW": 337.5
}

for df in [train_features, test_features]:

    angle = df["wd"].map(wind_angles)

    df["wd_sin"] = np.sin(np.deg2rad(angle))
    df["wd_cos"] = np.cos(np.deg2rad(angle))


# ============================================================
# 8. MISSINGNESS INDICATORS
# ============================================================

measurement_columns = [
    "current_PM2_5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM",
    "wd"
]

for df in [train_features, test_features]:

    for col in measurement_columns:
        df[f"{col}_missing"] = (
            df[col].isna().astype(int)
        )


# ============================================================
# 9. DATA-QUALITY FLAGS
# ============================================================

for df in [train_features, test_features]:

    df["pm25_floor_flag"] = (
        df["current_PM2_5"] == 3
    ).astype(int)

    df["pm10_equals_pm25"] = (
        df["PM10"] == df["current_PM2_5"]
    ).astype(int)

    df["pm25_gt_pm10"] = (
        df["current_PM2_5"] > df["PM10"]
    ).astype(int)

    df["o3_extreme"] = (
        df["O3"] > 400
    ).astype(int)

    df["co_floor"] = (
        df["CO"] == 100
    ).astype(int)

    df["co_ceiling"] = (
        df["CO"] == 10000
    ).astype(int)


# ============================================================
# 10. USEFUL PHYSICAL DERIVED FEATURES
# ============================================================

for df in [train_features, test_features]:

    # Dew-point spread
    df["TEMP_DEWP_diff"] = (
        df["TEMP"] - df["DEWP"]
    )

    # PM relationship
    df["PM10_PM25_ratio"] = (
        df["PM10"] /
        df["current_PM2_5"].replace(0, np.nan)
    )

    # Pollutant ratios
    df["NO2_CO_ratio"] = (
        df["NO2"] /
        df["CO"].replace(0, np.nan)
    )

    df["SO2_CO_ratio"] = (
        df["SO2"] /
        df["CO"].replace(0, np.nan)
    )


# ============================================================
# 11. TEMPORAL FEATURES
# ============================================================

# IMPORTANT:
# Temporal features are constructed using ONLY information
# available at or before the current observation time.
#
# We use timestamp-based lookup rather than simple shift()
# because some station-hours are missing from the dataset.


# ------------------------------------------------------------
# 11.1 Prepare temporal source data
# ------------------------------------------------------------

# Keep the original current PM2.5 values as the main
# time-series variable.

temporal_columns = [
    "current_PM2_5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM"
]


# ------------------------------------------------------------
# 11.2 Helper function for true time-based lags
# ------------------------------------------------------------

def add_time_lags(df, columns, lags):

    # Work on a copy so the original dataframe is preserved
    result = df.copy()

    # Create a lookup table indexed by
    # station + timestamp
    lookup = (
        df[["station", "observation_timestamp"] + columns]
        .set_index(["station", "observation_timestamp"])
        .sort_index()
    )

    for col in columns:

        for lag in lags:

            # Shift the timestamp BACKWARD by `lag` hours.
            #
            # Example:
            # current time = 10:00
            # lag_1h looks for value at 09:00
            shifted_time = (
                df["observation_timestamp"]
                - pd.Timedelta(hours=lag)
            )

            keys = pd.MultiIndex.from_arrays(
                [
                    df["station"].values,
                    shifted_time.values
                ],
                names=["station", "observation_timestamp"]
            )

            result[f"{col}_lag_{lag}h"] = (
                lookup[col]
                .reindex(keys)
                .to_numpy()
            )

    return result


# ------------------------------------------------------------
# 11.3 PM2.5 lags
# ------------------------------------------------------------

lags = [
    1,
    2,
    3,
    6,
    12,
    24,
    48,
    72
]

train_features = add_time_lags(
    train_features,
    columns=["current_PM2_5"],
    lags=lags
)

test_features = add_time_lags(
    test_features,
    columns=["current_PM2_5"],
    lags=lags
)


# ------------------------------------------------------------
# 11.4 Pollutant lags
# ------------------------------------------------------------

pollutant_columns = [
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3"
]

pollutant_lags = [
    1,
    3,
    6,
    24
]

train_features = add_time_lags(
    train_features,
    columns=pollutant_columns,
    lags=pollutant_lags
)

test_features = add_time_lags(
    test_features,
    columns=pollutant_columns,
    lags=pollutant_lags
)


# ------------------------------------------------------------
# 11.5 Weather lags
# ------------------------------------------------------------

weather_columns = [
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM"
]

weather_lags = [
    1,
    3,
    6,
    24
]

train_features = add_time_lags(
    train_features,
    columns=weather_columns,
    lags=weather_lags
)

test_features = add_time_lags(
    test_features,
    columns=weather_columns,
    lags=weather_lags
)


# ============================================================
# 12. PM2.5 CHANGE / TREND FEATURES
# ============================================================

for df in [train_features, test_features]:

    # 1-hour change
    df["PM25_change_1h"] = (
        df["current_PM2_5"]
        - df["current_PM2_5_lag_1h"]
    )

    # 3-hour change
    df["PM25_change_3h"] = (
        df["current_PM2_5"]
        - df["current_PM2_5_lag_3h"]
    )

    # 6-hour change
    df["PM25_change_6h"] = (
        df["current_PM2_5"]
        - df["current_PM2_5_lag_6h"]
    )

    # 24-hour change
    df["PM25_change_24h"] = (
        df["current_PM2_5"]
        - df["current_PM2_5_lag_24h"]
    )

    # Relative 1-hour change
    df["PM25_pct_change_1h"] = (
        df["PM25_change_1h"]
        / df["current_PM2_5_lag_1h"].replace(0, np.nan)
    )

    # Relative 24-hour change
    df["PM25_pct_change_24h"] = (
        df["PM25_change_24h"]
        / df["current_PM2_5_lag_24h"].replace(0, np.nan)
    )


# ============================================================
# 13. ROLLING PM2.5 FEATURES
# ============================================================

def add_rolling_features(df):

    result = df.copy()

    # Keep original row order so we can restore it later
    result["_original_order"] = np.arange(len(result))

    # Make sure data is chronologically ordered
    result = result.sort_values(
        ["station", "observation_timestamp"]
    )

    rolling_windows = [3, 6, 12, 24, 48]

    for window in rolling_windows:

        # ----------------------------------------------------
        # Calculate rolling statistics separately by station
        # ----------------------------------------------------

        rolling = (
            result
            .set_index("observation_timestamp")
            .groupby("station")["current_PM2_5"]
            .rolling(
                f"{window}h",
                closed="left",
                min_periods=1
            )
        )

        rolling_stats = rolling.agg(
            ["mean", "std", "min", "max"]
        ).reset_index()

        # Rename columns
        rolling_stats = rolling_stats.rename(
            columns={
                "mean": f"PM25_roll_mean_{window}h",
                "std": f"PM25_roll_std_{window}h",
                "min": f"PM25_roll_min_{window}h",
                "max": f"PM25_roll_max_{window}h"
            }
        )

        # ----------------------------------------------------
        # Merge using station + timestamp
        # ----------------------------------------------------

        result = result.merge(
            rolling_stats,
            on=["station", "observation_timestamp"],
            how="left",
            sort=False
        )

    # Restore original ordering
    result = (
        result
        .sort_values("_original_order")
        .drop(columns="_original_order")
        .reset_index(drop=True)
    )

    return result


train_features = add_rolling_features(train_features)
test_features = add_rolling_features(test_features)


# ============================================================
# 14. TEMPORAL DATA-AVAILABILITY FLAGS
# ============================================================

for df in [train_features, test_features]:

    # Tell the model whether recent history actually exists.

    df["PM25_lag_1h_missing"] = (
        df["current_PM2_5_lag_1h"].isna()
    ).astype(int)

    df["PM25_lag_24h_missing"] = (
        df["current_PM2_5_lag_24h"].isna()
    ).astype(int)

    df["PM25_history_available"] = (
        df[
            [
                "current_PM2_5_lag_1h",
                "current_PM2_5_lag_3h",
                "current_PM2_5_lag_6h",
                "current_PM2_5_lag_24h"
            ]
        ]
        .notna()
        .sum(axis=1)
    )


# ============================================================
# 15. CROSS-STATION CURRENT-TIME FEATURES
# ============================================================

# These use OTHER STATIONS at the SAME timestamp.
#
# They do NOT use t+1 information.
#
# This is potentially very powerful because your EDA showed
# strong correlation between stations.

for df in [train_features, test_features]:

    station_pm25 = (
        df.groupby("observation_timestamp")["current_PM2_5"]
    )

    df["network_PM25_mean"] = (
        station_pm25.transform("mean")
    )

    df["network_PM25_median"] = (
        station_pm25.transform("median")
    )

    df["network_PM25_std"] = (
        station_pm25.transform("std")
    )

    df["network_PM25_min"] = (
        station_pm25.transform("min")
    )

    df["network_PM25_max"] = (
        station_pm25.transform("max")
    )

    # Difference between this station and the network
    df["PM25_vs_network_mean"] = (
        df["current_PM2_5"]
        - df["network_PM25_mean"]
    )


# ============================================================
# 16. FINAL FEATURE SET
# ============================================================

X_train = train_features.copy()
X_test = test_features.copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (360954, 134)
X_test shape: (51063, 134)


In [3]:
# ============================================================
# INTERACTION FEATURES
# ============================================================

def add_interaction_features(df):
    df = df.copy()

    # --------------------------------------------------------
    # 1. PM2.5 × pollutants
    # --------------------------------------------------------
    df["PM25_x_PM10"] = (
        df["current_PM2_5"] * df["PM10"]
    )

    df["PM25_x_CO"] = (
        df["current_PM2_5"] * df["CO"]
    )

    df["PM25_x_NO2"] = (
        df["current_PM2_5"] * df["NO2"]
    )

    df["PM25_x_SO2"] = (
        df["current_PM2_5"] * df["SO2"]
    )

    df["PM25_x_O3"] = (
        df["current_PM2_5"] * df["O3"]
    )


    # --------------------------------------------------------
    # 2. PM2.5 × weather
    # --------------------------------------------------------
    df["PM25_x_WSPM"] = (
        df["current_PM2_5"] * df["WSPM"]
    )

    df["PM25_x_TEMP"] = (
        df["current_PM2_5"] * df["TEMP"]
    )

    df["PM25_x_DEWP"] = (
        df["current_PM2_5"] * df["DEWP"]
    )

    df["PM25_x_PRES"] = (
        df["current_PM2_5"] * df["PRES"]
    )

    df["PM25_x_RAIN"] = (
        df["current_PM2_5"] * df["RAIN"]
    )


    # --------------------------------------------------------
    # 3. PM10 × weather
    # --------------------------------------------------------
    df["PM10_x_WSPM"] = (
        df["PM10"] * df["WSPM"]
    )

    df["PM10_x_TEMP"] = (
        df["PM10"] * df["TEMP"]
    )

    df["PM10_x_DEWP"] = (
        df["PM10"] * df["DEWP"]
    )

    df["PM10_x_RAIN"] = (
        df["PM10"] * df["RAIN"]
    )


    # --------------------------------------------------------
    # 4. Pollutant × pollutant
    # --------------------------------------------------------
    df["CO_x_NO2"] = (
        df["CO"] * df["NO2"]
    )

    df["CO_x_SO2"] = (
        df["CO"] * df["SO2"]
    )

    df["NO2_x_SO2"] = (
        df["NO2"] * df["SO2"]
    )

    df["NO2_x_O3"] = (
        df["NO2"] * df["O3"]
    )


    # --------------------------------------------------------
    # 5. PM2.5 × network conditions
    # --------------------------------------------------------
    df["PM25_x_network_mean"] = (
        df["current_PM2_5"] * df["network_PM25_mean"]
    )

    df["PM25_x_network_std"] = (
        df["current_PM2_5"] * df["network_PM25_std"]
    )


    # --------------------------------------------------------
    # 6. Floor-regime interactions
    # --------------------------------------------------------
    # pm25_floor_flag should already exist:
    # 1 if current PM2.5 == 3, otherwise 0

    df["PM10_x_pm25_floor"] = (
        df["PM10"] * df["pm25_floor_flag"]
    )

    df["CO_x_pm25_floor"] = (
        df["CO"] * df["pm25_floor_flag"]
    )

    df["NO2_x_pm25_floor"] = (
        df["NO2"] * df["pm25_floor_flag"]
    )

    df["WSPM_x_pm25_floor"] = (
        df["WSPM"] * df["pm25_floor_flag"]
    )

    # --------------------------------------------------------
    # 6. Wind speed interactions
    # --------------------------------------------------------
    df["WSPM_x_network_mean"] = (
        df["WSPM"] * df["network_PM25_mean"]
    )

    df["WSPM_x_network_std"] = (
        df["WSPM"] * df["network_PM25_std"]
    )

        # --------------------------------------------------------
    # 7. Dew point depression interactions
    # --------------------------------------------------------
    # Reuses existing TEMP_DEWP_diff (= TEMP - DEWP) rather than
    # creating a duplicate "ddd" column.

    df["PM25_x_ddd"] = (
        df["current_PM2_5"] * df["TEMP_DEWP_diff"]
    )

    df["PM10_x_ddd"] = (
        df["PM10"] * df["TEMP_DEWP_diff"]
    )

    # --------------------------------------------------------
    # 8. PM2.5 / PM10 ratio
    # --------------------------------------------------------
    # Reciprocal of existing PM10_PM25_ratio — not a duplicate,
    # since 1/x is a nonlinear transform. Guard PM10 == 0.

    df["PM25_over_PM10"] = (
        df["current_PM2_5"] /
        df["PM10"].replace(0, np.nan)
    )

    return df

In [4]:
X_train = add_interaction_features(X_train)
X_test = add_interaction_features(X_test)

In [5]:
features_to_drop = [
    # Obvious interaction redundancy
    "PM25_x_PRES",

    # Calendar redundancy
    "day",
    "dayofyear",
    "weekofyear",

    # Pressure redundancy
    "PRES",
    "PRES_lag_1h",
    "PRES_lag_3h",
    "PRES_lag_6h",

    # Temperature redundancy
    "TEMP_lag_1h",
    "TEMP_lag_3h",
    "TEMP_lag_6h",

    # Dew point redundancy
    "DEWP_lag_1h",
    "DEWP_lag_3h",
    "DEWP_lag_6h",

    # Network redundancy
    "network_PM25_median",
    "network_PM25_min",
    "network_PM25_max",

    # Very long PM2.5 lag
    "current_PM2_5_lag_72h",

    # Long-window rolling maxima
    "PM25_roll_max_6h",
    "PM25_roll_max_12h",
    "PM25_roll_max_24h",
    "PM25_roll_max_48h",

    # O3 lags
    "O3_lag_1h",
    "O3_lag_3h",
    "O3_lag_6h",
    "O3_lag_24h",

    # Rain lags
    "RAIN_lag_1h",
    "RAIN_lag_3h",
    "RAIN_lag_6h",
    "RAIN_lag_24h",

    # Very weak interactions
    "PM25_x_RAIN",
    "PM10_x_RAIN",

    # Weak ratio
    "SO2_CO_ratio",

    # Redundant missingness indicators
    "TEMP_missing",
    "PRES_missing",
    "DEWP_missing",
    "RAIN_missing",
]


# ============================================================
# 2. SELECT FEATURES
# ============================================================
# observation_timestamp:
#   We already extracted useful time information such as
#   year, month, hour, dayofweek, etc.
#
# wd:
#   We converted wind direction into wd_sin and wd_cos.
#   Therefore the original string column is no longer needed.

selected_features = [
    f for f in X_train.columns
    if f not in features_to_drop
    and f not in ["observation_timestamp", "wd"]
]


# ============================================================
# 3. CREATE SELECTED DATASETS
# ============================================================

X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()


print("Selected features:", len(selected_features))
print("X_train:", X_train_selected.shape)
print("X_test:", X_test_selected.shape)


# ============================================================
# 4. DEFINE FEATURE TYPES
# ============================================================
# Only station remains categorical.
#
# wd has already been replaced by wd_sin and wd_cos.
# observation_timestamp has already been removed.

categorical_features = ["station"]

numeric_features_selected = [
    f for f in selected_features
    if f not in categorical_features
]


print("Numeric:", len(numeric_features_selected))
print("Categorical:", len(categorical_features))


# ============================================================
# 5. SANITY CHECK
# ============================================================

# Check that every feature in the pipeline actually exists
# in X_train_selected.

missing_numeric = [
    f for f in numeric_features_selected
    if f not in X_train_selected.columns
]

missing_categorical = [
    f for f in categorical_features
    if f not in X_train_selected.columns
]

print("Missing numeric features:", missing_numeric)
print("Missing categorical features:", missing_categorical)


# Check that no non-numeric columns accidentally entered
# the numeric pipeline.

non_numeric = X_train_selected[
    numeric_features_selected
].select_dtypes(exclude=np.number).columns.tolist()

print("Non-numeric features in numeric pipeline:", non_numeric)


Selected features: 124
X_train: (360954, 124)
X_test: (51063, 124)
Numeric: 123
Categorical: 1
Missing numeric features: []
Missing categorical features: []
Non-numeric features in numeric pipeline: []


In [6]:
X_train_selected.to_parquet('X_train.parquet', index=False)
X_test_selected.to_parquet('X_test.parquet', index=False)